In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [3]:
query = "버터떡이 뭐지?"

In [4]:
response = model.invoke(query)
response

AIMessage(content="**버터떡 (Butter Tteok)**은 말 그대로 '버터(butter)'와 '떡(tteok)'이 결합된 형태의 떡입니다. 전통적인 떡에 서양의 식재료인 버터를 더해 풍미를 끌어올린 퓨전 떡이라고 할 수 있습니다.\n\n**주요 특징:**\n\n1.  **풍부한 버터 풍미:** 쫄깃한 떡의 식감에 버터의 고소하고 풍부한 향, 그리고 은은한 단맛 또는 짭짤한 맛이 더해져 독특한 매력을 가집니다.\n2.  **다양한 제조 방식:**\n    *   **반죽에 버터 첨가:** 떡 반죽을 할 때 버터를 직접 섞어 쪄내는 방식입니다. 이렇게 하면 떡 자체에서 버터향이 배어 나옵니다.\n    *   **버터에 구워내기:** 미리 쪄낸 떡(주로 가래떡이나 절편 형태)을 프라이팬에 버터를 녹여 노릇하게 구워내는 방식입니다. 겉은 바삭하고 속은 쫄깃하며 버터향이 강하게 느껴집니다.\n    *   **버터 소스/코팅:** 쪄낸 떡 위에 녹인 버터와 설탕, 꿀 등을 섞은 소스를 바르거나 뿌려 코팅하는 방식입니다.\n3.  **맛과 식감:** 기본적으로 떡의 쫄깃한 식감은 유지하면서, 버터 특유의 고소함과 부드러움이 더해져 좀 더 서구적인 디저트나 간식 같은 느낌을 줍니다. 설탕이나 소금으로 간을 하여 단짠단짠(달고 짠) 맛을 내기도 합니다.\n4.  **인기:** 최근 몇 년간 젊은 층을 중심으로 인기를 얻고 있는 간식이며, 특히 '버터' 맛 트렌드(예: 허니버터칩 등)와 맞물려 더욱 주목받았습니다. 카페나 베이커리, 떡집 등에서 종종 찾아볼 수 있습니다.\n\n요약하자면, 버터떡은 쫄깃한 한국의 떡과 고소한 서양의 버터가 만나 탄생한, 익숙하면서도 새로운 맛의 퓨전 간식입니다.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gem

In [5]:
from langchain_community.tools import DuckDuckGoSearchResults 

search = DuckDuckGoSearchResults(results_separator=';\n')
docs = search.invoke(query)

print(docs)

snippet: 6 days ago · 일반적으로, 〈버터〉라는 말은 다른 설명이 붙지 않으면 스프레드 유제품을 가리키는 말이지만, 보통 야채로 만든 퓌레 나 견과류 로 만든 땅콩 버터 나 아몬드 버터 도 가리키며, 사과 버터 …, title: 버터 - 위키백과, 우리 모두의 백과사전, link: https://ko.wikipedia.org/wiki/버터;
snippet: , title: 버터떡 - 나무위키, link: ;
snippet: 6 days ago · 국산 원유로 국내 제조한 발효 버터도 출시되어 있으나 가격이 일반 버터에 비해 비싸다. 서유럽산 버터, 특히 프랑스 · 덴마크 · 아일랜드 산 버터는 거의 다 발효 과정을 거친다. 버터 특유의 …, title: 버터 - 나무위키, link: https://namu.wiki/w/버터;
snippet: chapter. 건강 62개의 글 chapter. 건강목록열기 chapter. 건강 버터 종류 완벽 정리: 무염버터·가염버터·발효버터 차이, 효능, 건강하게 먹는 방법까지, title: 버터 종류 완벽 정리: 무염 버터 ·가염 버터 ·발효 버터 차이, 효능 ..., link: https://blog.naver.com/just_do_whatiwant/224210043807


### 🔎 웹 검색 도구

In [6]:
from langchain_core.tools import tool
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_community.tools import DuckDuckGoSearchResults

@tool
def get_web_search(query: str, search_period: str='w', region: str='kr-kr') -> str:
    """
    특정 지역과 기간을 설정하여 웹 검색(뉴스)을 수행합니다.

    Args:
        query (str): 검색어
        search_period (str): 검색 기간. 'd'(일간), 'w'(주간), 'm'(월간), 'y'(연간) 중 선택
        region (str): 검색 지역 코드. 한국('kr-kr'), 미국('us-en'), 일본('jp-jp'), 영국('uk-en') 등
    
    Returns:
        str: 검색된 뉴스 결과 리스트
    """
    # 1. 전달받은 region과 search_period로 API 설정
    wrapper = DuckDuckGoSearchAPIWrapper(
        region=region,
        time=search_period
    )

    print('\n----- WEB SEARCH (News) -----')
    print(f"QUERY  : {query}")
    print(f"REGION : {region}")
    print(f"PERIOD : {search_period}")

    search = DuckDuckGoSearchResults(
        api_wrapper=wrapper,
        backend="news",
        results_separator=';\n'
    )

    searched = search.invoke(query)
    return searched

In [7]:
from langchain.agents import create_agent

# 도구들을 tools 리스트에 추가
tools = [get_web_search,]

# 에이전트 생성
agent = create_agent(
    model,
    tools=tools,
    system_prompt="너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."
)

In [8]:
query

'버터떡이 뭐지?'

In [9]:
# Run the agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)


----- WEB SEARCH (News) -----
QUERY  : 버터떡이 뭐야
REGION : kr-kr
PERIOD : w


In [10]:
result

{'messages': [HumanMessage(content='버터떡이 뭐지?', additional_kwargs={}, response_metadata={}, id='82a7cbf6-fcc8-4290-b5a2-820b3cbc96f0'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_web_search', 'arguments': '{"query": "\\ubc84\\ud130\\ub5a1\\uc774 \\ubb50\\uc57c"}'}, '__gemini_function_call_thought_signatures__': {'68b6509c-f8c7-476c-bf8d-3d16b1421cba': 'CokCAb4+9vsFOVBY5h2i9/NdK1/ppcNFKGqtTlTV2lCxPiDa1mVLmgLqifYhnpQ+vaD6dAP/6RQeULYnAgeM7aJw/y7f6rWYnfc+Q1eDM4nbRzfQp5S6PVrqCrTkGHYyfhzRNdLEYLZn71fLU9+hGfHKv5czyQQOXlvpNUVP6rhzbbfqg+1NHbtGOm0iQ/UvQmk0PEfKyMJF3/zan9Z7FV6CSEznrge1ktdd32gyuPe15ELxa+7+9dXmaAmVWiAdsZRuS6mtzkErVcrE7+z9NtP/qgZKMbtJ16A0nKJbxUoLiqmgdOPrwACg+Gs1bJvEvXdVsBYC8WtU2jwNSL0Mb0z+opFJ7MreCXSXjg=='}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--832f12d3-85e8-4cb0-9d59-c4448eb3c54a-0', 